<a href="https://colab.research.google.com/github/ekaratnida/Applied-machine-learning/blob/master/HandWriting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#/content/drive/MyDrive/Course/DADS6003_Applied-machine-learning/week5_logistic/hand-writing

In [2]:
import os
import cv2
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

base_dir = "/content/drive/MyDrive/Course/DADS6003_Applied-machine-learning/week5_logistic/hand-writing"

image_data = []
labels = []

for digit in range(10):

    folder_path = os.path.join(base_dir, str(digit))

    if os.path.exists(folder_path) and os.path.isdir(folder_path):

        print(f"Processing folder: {folder_path}")

        for filename in os.listdir(folder_path):
            if filename.lower().endswith('.png'):
                img_path = os.path.join(folder_path, filename)
                try:
                    # Read image in grayscale
                    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

                    if img is not None:
                        img = cv2.resize(img, (28, 28))
                        pixels = img.flatten() #784
                        image_data.append(pixels)
                        labels.append(digit)
                    else:
                        print(f"Warning: Could not read image {img_path}")
                except Exception as e:
                    print(f"Error processing {img_path}: {e}")
    else:
        print(f"Warning: Folder for digit {digit} not found at {folder_path}. Skipping.")


Processing folder: /content/drive/MyDrive/Course/DADS6003_Applied-machine-learning/week5_logistic/hand-writing/0
Processing folder: /content/drive/MyDrive/Course/DADS6003_Applied-machine-learning/week5_logistic/hand-writing/1
Processing folder: /content/drive/MyDrive/Course/DADS6003_Applied-machine-learning/week5_logistic/hand-writing/2
Processing folder: /content/drive/MyDrive/Course/DADS6003_Applied-machine-learning/week5_logistic/hand-writing/3
Processing folder: /content/drive/MyDrive/Course/DADS6003_Applied-machine-learning/week5_logistic/hand-writing/4
Processing folder: /content/drive/MyDrive/Course/DADS6003_Applied-machine-learning/week5_logistic/hand-writing/5
Processing folder: /content/drive/MyDrive/Course/DADS6003_Applied-machine-learning/week5_logistic/hand-writing/6
Processing folder: /content/drive/MyDrive/Course/DADS6003_Applied-machine-learning/week5_logistic/hand-writing/7
Processing folder: /content/drive/MyDrive/Course/DADS6003_Applied-machine-learning/week5_logisti

In [ ]:
# Create DataFrame if images were processed
if image_data:
    X_pixels = np.array(image_data)
    print(X_pixels.shape)
    Y_labels = np.array(labels)
    print(Y_labels)

    # Create column names for pixels (e.g., pixel_0, pixel_1, ..., pixel_783 for 28x28 images)
    pixel_col_names = [f'pixel_{i}' for i in range(X_pixels.shape[1])]

    # Create the DataFrame with pixels as X and folder name as Y
    df = pd.DataFrame(X_pixels, columns=pixel_col_names)
    df['Y'] = Y_labels

    print("\nDataFrame created successfully:")
    print(df.head())
    print("\nDataFrame Info:")
    df.info()
    print(f"\nShape of DataFrame: {df.shape}")
else:
    print("No PNG files found or processed. DataFrame was not created.")


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Separate features (X) and target (Y)
X = df.drop('Y', axis=1) # Features are all columns except 'Y'
Y = df['Y'] # Target is the 'Y' column

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.5, random_state=42, stratify=Y)

print("Data split into training and testing sets:")
print(f"X_train shape: {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Y_test shape: {Y_test.shape}")

# Create a Logistic Regression model
# Set max_iter to a higher value to ensure convergence for larger datasets, and solver='liblinear' for good performance on small to medium datasets.
model = LogisticRegression(max_iter=1000, solver='liblinear')

# Train the model
model.fit(X_train, Y_train)

print("\nLogistic Regression model created and trained successfully.")

Data split into training and testing sets:
X_train shape: (10, 784)
Y_train shape: (10,)
X_test shape: (10, 784)
Y_test shape: (10,)

Logistic Regression model created and trained successfully.


In [13]:
print(model.coef_.shape)
print(model.intercept_.shape)

(10, 784)
(10,)


In [16]:
from sklearn.metrics import accuracy_score

# Make predictions on the test set
Y_pred = model.predict(X_test)
print(Y_pred)
Y_pred_prob = model.predict_proba(X_test)
print(Y_pred_prob)

# Calculate the accuracy
accuracy = accuracy_score(Y_test, Y_pred)

print(f"\nModel Accuracy on Test Set: {accuracy:.4f}")

[9 2 0 6 8 5 3 7 1 4]
[[2.30093364e-06 8.58425028e-07 4.91022613e-06 3.99792198e-06
  3.40908512e-06 5.45739706e-06 3.97437607e-06 2.45187917e-08
  1.18989976e-05 9.99963168e-01]
 [1.51897188e-06 7.42861534e-06 9.99953770e-01 6.97526005e-06
  4.01944079e-08 2.56345719e-06 2.96152552e-06 2.05150113e-05
  3.57574454e-07 3.86968260e-06]
 [9.99974448e-01 6.28909365e-07 1.53006484e-06 1.08216037e-06
  2.83620114e-06 6.73078529e-06 4.40323515e-06 4.93451520e-06
  2.75630645e-07 3.13040423e-06]
 [2.02636803e-06 4.19938072e-06 1.71309705e-06 2.97965598e-06
  2.27953994e-06 3.74769724e-06 9.99978515e-01 2.06925881e-07
  1.99178699e-06 2.34075510e-06]
 [5.30094370e-07 9.59621933e-07 6.24836539e-07 2.74877239e-06
  2.17395713e-06 2.44986137e-06 5.49613180e-06 2.56086782e-06
  9.99971023e-01 1.14331863e-05]
 [5.29671858e-06 2.14333910e-07 2.76664744e-06 8.77557333e-06
  5.69476953e-07 9.99966146e-01 6.41350765e-06 3.65015242e-06
  1.17189810e-06 4.99525062e-06]
 [9.43541247e-07 1.89637193e-06 7.57

In [17]:
import os
import cv2
import numpy as np

# Define the directory where new test images are located
# This path assumes 'test' is a subfolder within your base_dir
new_test_images_dir = os.path.join(base_dir, 'test')

# Create the directory if it doesn't exist, and instruct the user
if not os.path.exists(new_test_images_dir):
    os.makedirs(new_test_images_dir)
    print(f"Created directory: {new_test_images_dir}")
    print("Please place your test PNG images in this directory to continue.")
else:
    print(f"Loading test images from: {new_test_images_dir}")

    test_image_data = []
    test_image_filenames = []

    # Iterate through images in the new test directory
    for filename in os.listdir(new_test_images_dir):
        if filename.lower().endswith('.png'):
            img_path = os.path.join(new_test_images_dir, filename)
            try:
                # Read image in grayscale
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

                if img is not None:
                    img = cv2.resize(img, (28, 28))
                    pixels = img.flatten()
                    test_image_data.append(pixels)
                    test_image_filenames.append(filename)
                else:
                    print(f"Warning: Could not read image {img_path}. Skipping.")
            except Exception as e:
                print(f"Error processing {img_path}: {e}. Skipping.")

    if test_image_data:
        # Convert list of pixel arrays to a NumPy array
        X_new_test = np.array(test_image_data)

        # Make predictions using the trained model
        new_predictions = model.predict(X_new_test)

        print("\nPredictions for new test images:")
        for i, filename in enumerate(test_image_filenames):
            print(f"Image: {filename}, Predicted Digit: {new_predictions[i]}")
    else:
        print("No PNG images found in the test directory to make predictions.")

Loading test images from: /content/drive/MyDrive/Course/DADS6003_Applied-machine-learning/week5_logistic/hand-writing/test

Predictions for new test images:
Image: ekarat_test_1.png, Predicted Digit: 1


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
